# End-to-end ISP+CNN training (Colab)

Thin Colab wrapper around `scripts/run_e2e_train.py`. Trains the differentiable ISP and the residual CNN jointly with the quality-aligned surrogate loss. The ISP exposes its trainable `nn.Parameter` tensors (CCM, gamma, plus the LTM, AWB, histogram, sharpening, denoise, and RAW-Y blend coefficients); the residual CNN's weights are loaded from the pretrained warm-start checkpoint.

Best checkpoint is selected by the normalized composite `VIF_norm + a*NRQM_norm + b*UNIQUE_norm` on the per-scene mini-val split.

## Required Google Drive files

Create this folder:

`MyDrive/ISP_colab/e2e_quality/`

Upload these files:

- `e2e_quality_bundle.zip`
- `train_patches.h5`
- `cnn_pretrained.pth` (output of the warm-start notebook)
- `day_val_mini.bin`
- `day_val_mini.yuv`
- `night_val_mini.bin`
- `night_val_mini.yuv`

Outputs are written to:

`MyDrive/ISP_colab/e2e_quality_outputs/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ISP_colab')
DRIVE_INPUT = DRIVE_ROOT / 'e2e_quality'
DRIVE_OUTPUT = DRIVE_ROOT / 'e2e_quality_outputs'

LOCAL_REPO = Path('/content/ISP')
LOCAL_TRAIN_H5 = LOCAL_REPO / 'dataset' / 'train_patches.h5'
LOCAL_CKPTS = LOCAL_REPO / 'artifacts' / 'checkpoints'

CLEAR_OLD_OUTPUTS = True

EPOCHS = 15
BATCH_SIZE = 4
LR_ISP = 1e-6
LR_CNN = 5e-6
LAMBDA_UV = 1.0         
LOSS = 'quality'
BEST_CRITERION = 'composite'
W_SSIM = 1.0
W_VIF = 5.0            
W_UNIQUE = 0.3
W_UV = 1.0
W_L1_Y = 0.0            

W_VIF_FLOOR = 50.0
VIF_TARGET = 0.7
W_UNIQUE_FLOOR = 10.0
UNIQUE_TARGET = 0.0

EVAL_EVERY = 1
EVAL_MAX_FRAMES = 3
CHECKPOINT_EVERY = 1
SEED = 42
NUM_WORKERS = 2

ISP_REG_WEIGHT = 0.05
ISP_REG_CCM_SCALE = 0.05
ISP_REG_GAMMA_SCALE = 0.1
ISP_REG_CONTINUOUS_SCALE = 0.1
ISP_REG_GAIN_SCALE = 10.0
ISP_REG_EPS_LOG_SCALE = 1.0

VIF_GUARD_RATIO = 0.9
VIF_GUARD_PENALTY = 10.0

SCENE_AWARE_TRAIN = True
BALANCE_SCENES = True
DAY_LOSS_WEIGHT = 1.0
NIGHT_LOSS_WEIGHT = 2.0

RESUME_MODE = 'never'    # 'never' | 'auto' | 'force'
MAX_TRAIN_BATCHES = None

print(f'Drive input:  {DRIVE_INPUT}')
print(f'Drive output: {DRIVE_OUTPUT}')
print(f'Local repo:   {LOCAL_REPO}')

## 2. Check GPU

In [ ]:
import subprocess
import torch

try:
    print(subprocess.check_output([
        'nvidia-smi',
        '--query-gpu=name,memory.total,driver_version',
        '--format=csv',
    ]).decode())
except Exception as exc:
    print(f'nvidia-smi is not available: {exc}')

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('Please switch Colab runtime to GPU before starting end-to-end training.')

## 3. Install dependencies

In [ ]:
!pip install -q pyiqa tomli h5py tqdm pillow

## 4. Prepare local workspace

Extracts the code bundle, copies `train_patches.h5` and the warm-start checkpoint to local SSD, and the four mini-val files into `data/`.

In [ ]:
import os
import shutil
import time
import zipfile

def required(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return path

def required_one(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    listed = ', '.join(str(Path(p)) for p in paths)
    raise FileNotFoundError(f'Missing required file (looked for one of: {listed})')

code_zip = DRIVE_INPUT / 'e2e_quality_bundle.zip'
train_h5 = required(DRIVE_INPUT / 'train_patches.h5')
ckpt = required(DRIVE_INPUT / 'cnn_pretrained.pth')
mini_files = {
    name: required(DRIVE_INPUT / name)
    for name in ['day_val_mini.bin', 'day_val_mini.yuv',
                 'night_val_mini.bin', 'night_val_mini.yuv']
}

if LOCAL_REPO.exists():
    print(f'Removing old local workspace: {LOCAL_REPO}')
    shutil.rmtree(LOCAL_REPO)
LOCAL_REPO.mkdir(parents=True, exist_ok=True)

if CLEAR_OLD_OUTPUTS and DRIVE_OUTPUT.exists():
    assert DRIVE_OUTPUT.name == 'e2e_quality_outputs'
    assert str(DRIVE_OUTPUT).startswith(str(DRIVE_ROOT))
    print(f'Removing old training outputs: {DRIVE_OUTPUT}')
    shutil.rmtree(DRIVE_OUTPUT)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f'Unzipping {code_zip.name}...', end=' ', flush=True)
t0 = time.time()
with zipfile.ZipFile(code_zip, 'r') as zf:
    zf.extractall(LOCAL_REPO)
print(f'done ({time.time() - t0:.1f}s)')

LOCAL_TRAIN_H5.parent.mkdir(parents=True, exist_ok=True)
print(f'Copying train_patches.h5...', end=' ', flush=True)
t0 = time.time()
shutil.copy2(train_h5, LOCAL_TRAIN_H5)
size_gb = LOCAL_TRAIN_H5.stat().st_size / (1024 ** 3)
print(f'done ({size_gb:.2f} GB, {time.time() - t0:.1f}s)')

LOCAL_CKPTS.mkdir(parents=True, exist_ok=True)
shutil.copy2(ckpt, LOCAL_CKPTS / 'cnn_pretrained.pth')
print(f'Copied warm-start checkpoint: {LOCAL_CKPTS / "cnn_pretrained.pth"}')

(LOCAL_REPO / 'data').mkdir(parents=True, exist_ok=True)
for name, src in mini_files.items():
    dst = LOCAL_REPO / 'data' / name
    shutil.copy2(src, dst)
    print(f'  copied {name}: {dst.stat().st_size / 1024**2:.1f} MB')

checks = [
    LOCAL_REPO / 'scripts' / 'run_e2e_train.py',
    LOCAL_REPO / 'isp' / 'training' / 'training_utils.py',
    LOCAL_REPO / 'isp' / 'training' / 'quality_loss.py',
    LOCAL_REPO / 'metrics' / 'vif.py',
    LOCAL_REPO / 'data' / 'imx623.toml',
    LOCAL_REPO / 'dataset' / 'splits_mini.json',
    LOCAL_REPO / 'artifacts' / 'baselines' / 'norm_weights.json',
    LOCAL_REPO / 'artifacts' / 'checkpoints' / 'cnn_pretrain' / 'pretrain_eval_metrics.json',
    LOCAL_TRAIN_H5,
    LOCAL_CKPTS / 'cnn_pretrained.pth',
]
for path in checks:
    print(f'{path.relative_to(LOCAL_REPO)}: {"OK" if path.exists() else "MISSING"}')
    if not path.exists():
        raise FileNotFoundError(path)

bundle_markers = [
    (LOCAL_REPO / 'scripts' / 'run_e2e_train.py', 'sanitize_trainable_params_dict'),
    (LOCAL_REPO / 'scripts' / 'run_e2e_train.py', 'load_e2e_minival_baseline'),
    (LOCAL_REPO / 'isp' / 'pipeline' / 'pipeline.py', 'project_trainable_params_'),
]
for path, marker in bundle_markers:
    text = path.read_text(encoding='utf-8')
    if marker not in text:
        raise RuntimeError(
            f'Stale bundle detected: {path.name} does not contain marker {marker!r}. '
            'Upload e2e_quality_bundle_fixed.zip or replace e2e_quality_bundle.zip with the rebuilt bundle.'
        )
print('Bundle self-check: OK (fixed eval/NaN guards present)')

print('\nAll end-to-end inputs are ready.')

## 5. Resume detection

Checks Drive for an existing `e2e_resume.pth`. With `RESUME_MODE = 'auto'` the training script picks it up and continues from the next epoch.

In [ ]:
resume_path = DRIVE_OUTPUT / 'e2e_resume.pth'
has_resume = resume_path.exists()
will_resume = (RESUME_MODE == 'force') or (RESUME_MODE == 'auto' and has_resume)

print(f'Resume mode:     {RESUME_MODE}')
print(f'Resume ckpt:     {resume_path}')
print(f'Exists on Drive: {has_resume}')
print(f'Will resume:     {will_resume}')

if has_resume:
    meta = torch.load(str(resume_path), map_location='cpu', weights_only=False)
    print(f'  last completed epoch: {meta["epoch"]}')
    print(f'  best_val_loss:        {meta["best_val_loss"]:.6f} @ epoch {meta["best_epoch"]}')
    print(f'  history entries:      {len(meta["history"])}')
    del meta
elif RESUME_MODE == 'force':
    raise RuntimeError(f'RESUME_MODE="force" but no resume checkpoint at {resume_path}.')

## 6. Run end-to-end training

In [ ]:
import sys

cmd = [
    sys.executable, '-B', 'scripts/run_e2e_train.py',
    '--device', 'cuda',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--lr-isp', str(LR_ISP),
    '--lr-cnn', str(LR_CNN),
    '--lambda-uv', str(LAMBDA_UV),
    '--loss', LOSS,
    '--best-criterion', BEST_CRITERION,
    '--w-ssim', str(W_SSIM),
    '--w-vif', str(W_VIF),
    '--w-unique', str(W_UNIQUE),
    '--w-l1-y', str(W_L1_Y),
    '--w-uv', str(W_UV),
    '--w-vif-floor', str(W_VIF_FLOOR),
    '--vif-target', str(VIF_TARGET),
    '--w-unique-floor', str(W_UNIQUE_FLOOR),
    '--unique-target', str(UNIQUE_TARGET),
    '--isp-reg-weight', str(ISP_REG_WEIGHT),
    '--isp-reg-ccm-scale', str(ISP_REG_CCM_SCALE),
    '--isp-reg-gamma-scale', str(ISP_REG_GAMMA_SCALE),
    '--isp-reg-continuous-scale', str(ISP_REG_CONTINUOUS_SCALE),
    '--isp-reg-gain-scale', str(ISP_REG_GAIN_SCALE),
    '--isp-reg-eps-log-scale', str(ISP_REG_EPS_LOG_SCALE),
    '--vif-guard-ratio', str(VIF_GUARD_RATIO),
    '--vif-guard-penalty', str(VIF_GUARD_PENALTY),
    '--eval-every', str(EVAL_EVERY),
    '--eval-max-frames', str(EVAL_MAX_FRAMES),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--seed', str(SEED),
    '--num-workers', str(NUM_WORKERS),
    '--day-loss-weight', str(DAY_LOSS_WEIGHT),
    '--night-loss-weight', str(NIGHT_LOSS_WEIGHT),
    '--train-h5', 'dataset/train_patches.h5',
    '--splits-json', 'dataset/splits_mini.json',
    '--config', 'data/imx623.toml',
    '--ckpt', 'artifacts/checkpoints/cnn_pretrained.pth',
    '--norm-weights', 'artifacts/baselines/norm_weights.json',
    '--output-dir', str(DRIVE_OUTPUT),
]
if SCENE_AWARE_TRAIN:
    cmd += ['--scene-aware-train']
if BALANCE_SCENES:
    cmd += ['--balance-scenes']
if MAX_TRAIN_BATCHES is not None:
    cmd += ['--max-train-batches', str(MAX_TRAIN_BATCHES)]
if will_resume:
    cmd += ['--resume']

print('Running:')
print(' '.join(cmd))
print()

proc = subprocess.Popen(
    cmd, cwd=str(LOCAL_REPO),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True,
)
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    if proc.poll() is None:
        proc.terminate()

print(f'\n[subprocess exit={ret}]')
if ret != 0:
    raise RuntimeError(f'End-to-end training exited with code {ret}')

## 7. Verify outputs on Drive

In [ ]:
import glob
files = sorted(glob.glob(f'{DRIVE_OUTPUT}/e2e_*'))
for f in files:
    size_mb = os.path.getsize(f) / 1024 ** 2
    print(f'  {os.path.basename(f):<40s}  {size_mb:>8.2f} MB')
print(f'\n{len(files)} e2e_* files on Drive.')

## 8. Training curves and ISP parameter trace

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open(f'{DRIVE_OUTPUT}/e2e_history.json', 'r') as f:
    history = json.load(f)

train_recs = [r for r in history if 'train_loss' in r]
val_recs = [r for r in train_recs if r.get('val_composite') is not None]

epochs_t = [r['epoch'] for r in train_recs]
train_loss = [r['train_loss'] for r in train_recs]

epochs_v = [r['epoch'] for r in val_recs]
val_vif  = [r['val_vif'] for r in val_recs]
val_nrqm = [r['val_nrqm'] for r in val_recs]
val_unique = [r['val_unique'] for r in val_recs]
val_composite = [r['val_composite'] for r in val_recs]

all_recs = [r for r in history if 'isp_params' in r]
epochs_p = [r['epoch'] for r in all_recs]
gammas   = [r['isp_params']['gamma']      for r in all_recs]
ccm_fro  = [float(np.linalg.norm(np.array(r['isp_params']['ccm_matrix']))) for r in all_recs]
ltm_a    = [r['isp_params'].get('ltm_a') for r in all_recs]
ltm_dg   = [r['isp_params'].get('ltm_detail_gain') for r in all_recs]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].plot(epochs_t, train_loss)
axes[0, 0].set_title('Train quality loss'); axes[0, 0].set_xlabel('epoch'); axes[0, 0].grid(alpha=0.3)

if val_recs:
    axes[0, 1].plot(epochs_v, val_composite, 'o-', label='composite')
    axes[0, 1].plot(epochs_v, val_vif, 's-', label='VIF')
    axes[0, 1].plot(epochs_v, val_unique, '^-', label='UNIQUE')
    axes[0, 1].set_title('Validation quality metrics'); axes[0, 1].set_xlabel('epoch')
    axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)
    axes[0, 2].plot(epochs_v, val_nrqm, '.-')
    axes[0, 2].set_title('Validation NRQM'); axes[0, 2].set_xlabel('epoch'); axes[0, 2].grid(alpha=0.3)

axes[1, 0].plot(epochs_p, gammas, '.-', label='gamma')
axes[1, 0].set_title('Trainable gamma'); axes[1, 0].set_xlabel('epoch'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(epochs_p, ccm_fro, '.-', color='tab:green')
axes[1, 1].set_title('CCM Frobenius norm'); axes[1, 1].set_xlabel('epoch'); axes[1, 1].grid(alpha=0.3)

if any(v is not None for v in ltm_a):
    axes[1, 2].plot(epochs_p, ltm_a, '.-', label='ltm_a')
    if any(v is not None for v in ltm_dg):
        ax2 = axes[1, 2].twinx()
        ax2.plot(epochs_p, ltm_dg, '.-', color='tab:orange', label='ltm_detail_gain')
        ax2.set_ylabel('ltm_detail_gain', color='tab:orange')
    axes[1, 2].set_title('LTM trainables'); axes[1, 2].set_xlabel('epoch'); axes[1, 2].legend(loc='upper left'); axes[1, 2].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(f'{DRIVE_OUTPUT}/e2e_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

if val_recs:
    best = max(val_recs, key=lambda r: r['val_composite'])
    print(f'\nBest-by-composite epoch: {best["epoch"]}')
    print(f'  composite={best["val_composite"]:.6f}')
    print(f'  val_vif={best["val_vif"]:.4f}  val_nrqm={best["val_nrqm"]:.4f}  val_unique={best["val_unique"]:.4f}')